In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


In [ ]:
import time
import pandas as pd
from textblob import TextBlob
from transformers import pipeline
from pathlib import Path


In [32]:
from functools import lru_cache

_lemmatizer = WordNetLemmatizer()


@lru_cache(maxsize=8)
def _stopword_set(language: str) -> frozenset:
    return frozenset(stopwords.words(language))


def clean_text(text: str, language: str = "english") -> str:
    """Lowercase, tokenize, drop stopwords and non-alpha tokens, lemmatize; return a single string.

    NLTK data required (run once if missing):
        import nltk
        nltk.download("punkt")
        nltk.download("stopwords")
        nltk.download("wordnet")
        nltk.download("omw-1.4")
    """
    if text is None:
        return ""
    s = str(text).strip().lower()
    if not s:
        return ""

    stops = _stopword_set(language)
    tokens = word_tokenize(s)
    out: list[str] = []
    for tok in tokens:
        if not tok.isalpha():
            continue
        if tok in stops:
            continue
        out.append(_lemmatizer.lemmatize(tok))
    return " ".join(out)

In [33]:
def load_excel(
    path: str | Path,
    sheet_name: str | int | list[int | str] | None = 0,
    header: int | None = 0,
    engine: str = "openpyxl",
) -> pd.DataFrame | dict[int | str, pd.DataFrame]:
    """Load one or more sheets from an `.xlsx` file.

    Parameters
    ----------
    path : str or Path
        Path to the workbook.
    sheet_name : str, int, list, or None
        Sheet to read: name, 0-based index, list of names/indices, or None for all sheets.
    header : int or None
        Row to use as column names (None = no header, e.g. single-column eval sheets).
    engine : str
        `openpyxl` for `.xlsx` (default).

    Returns
    -------
    DataFrame, or dict of DataFrames if multiple sheets were requested.
    """
    p = Path(path).expanduser().resolve()
    if not p.is_file():
        raise FileNotFoundError(p)
    return pd.read_excel(p, sheet_name=sheet_name, header=header, engine=engine)

In [34]:
HF_SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"


def _scores_to_hf_outputs(scores: list[dict]) -> tuple[float, float, float, float, str]:
    """Pipeline scores (top_k=None) → P(neg), P(neu), P(pos), polarity, argmax label."""
    m = {str(s["label"]).lower(): float(s["score"]) for s in scores}
    neg = m.get("negative", m.get("label_0", 0.0))
    neu = m.get("neutral", m.get("label_1", 0.0))
    pos = m.get("positive", m.get("label_2", 0.0))
    polarity = pos - neg
    label = max(
        (("negative", neg), ("neutral", neu), ("positive", pos)), key=lambda x: x[1]
    )[0]
    return neg, neu, pos, polarity, label


def textblob_subjectivity(text: str) -> float:
    """Objective (0) → subjective (1); empty input → 0."""
    if text is None or not str(text).strip():
        return 0.0
    return float(TextBlob(str(text)).sentiment.subjectivity)


def hf_sentiment_batch(
    texts: list[str],
    *,
    model_name: str = HF_SENTIMENT_MODEL,
    batch_size: int = 16,
) -> tuple[list[float], list[float], list[float], list[float], list[str]]:
    """HF class probabilities, polarity (pos−neg), and argmax label."""
    if not texts:
        return [], [], [], [], []
    _empty_mask = [not str(t).strip() for t in texts]
    _feed = [t if str(t).strip() else " " for t in texts]
    clf = pipeline(
        "sentiment-analysis",
        model=model_name,
        tokenizer=model_name,
        truncation=True,
        max_length=512,
        top_k=None,
    )
    rows = clf(_feed, batch_size=batch_size)
    probs_neg: list[float] = []
    probs_neu: list[float] = []
    probs_pos: list[float] = []
    polarities: list[float] = []
    labels: list[str] = []
    for i, row in enumerate(rows):
        if _empty_mask[i]:
            probs_neg.append(0.0)
            probs_neu.append(1.0)
            probs_pos.append(0.0)
            polarities.append(0.0)
            labels.append("neutral")
            continue
        neg, neu, pos, pol, lab = _scores_to_hf_outputs(row)
        probs_neg.append(neg)
        probs_neu.append(neu)
        probs_pos.append(pos)
        polarities.append(pol)
        labels.append(lab)
    return probs_neg, probs_neu, probs_pos, polarities, labels

In [35]:
# Load eval1kFinal; all outputs below go under this notebook's folder (3.3/)
import time
from pathlib import Path

_cwd = Path.cwd().resolve()
NB_DIR = next(
    (p for p in (_cwd, _cwd / "3.3", _cwd.parent / "3.3") if (p / "main.ipynb").is_file()),
    _cwd,
)

_input_candidates = [
    NB_DIR / "evalPersonalCrawl.xlsx",
    _cwd / "evalPersonalCrawl.xlsx",
    _cwd / "3.3" / "evalPersonalCrawl.xlsx",
    _cwd / "3.1new" / "3.1Final" / "evalPersonalCrawl.xlsx",
    _cwd.parent / "3.1new" / "3.1Final" / "evalPersonalCrawl.xlsx",
]
file_path = next((p for p in _input_candidates if p.is_file()), None)
if file_path is None:
    raise FileNotFoundError(
        "evalPersonalCrawl.xlsx not found. Place a copy in 3.3/ or run from repo root."
    )
file_path = file_path.resolve()

_raw = load_excel(
    file_path,
    sheet_name="eval1kFinal",
    header=None,
    engine="openpyxl",
)
if str(_raw.iloc[0, 0]).strip().lower() == "comment":
    df = _raw.iloc[1:].copy().reset_index(drop=True)
    df.columns = _raw.iloc[0].astype(str).tolist()
else:
    df = _raw
_original_cols = list(df.columns)
_comment_col = df["comment"] if "comment" in df.columns else df.iloc[:, 0]
texts = _comment_col.astype(str).tolist()

# --- 1. Preprocess the text ---
start_time = time.time()
cleaned_texts = [clean_text(text) for text in texts]
df["cleaned_comment"] = cleaned_texts
preprocessing_time = time.time() - start_time
print(f"Preprocessing Time: {preprocessing_time:.4f} seconds")


Preprocessing Time: 1.1014 seconds


In [36]:
df.head()

,comment,polarity,subjectivity,label,cleaned_comment
0,Climate change is the new military industrial ...,-0.078409,0.238636,negative,climate change new military industrial complex...
1,"All good point, but wild fires aren’t because ...",0.08,0.58,positive,good point wild fire climate change year fores...
2,"Or maybe I do. \n \nFrom my comment, it's h...",0.101412,0.55303,positive,maybe comment hard tell since authority blame ...
3,&gt; The people who make money from that disbe...,0.088343,0.488492,positive,gt people make money disbelief gt gt spend lot...
4,All climate doomsday predictions up until now ...,0.15,0.7,positive,climate doomsday prediction also debunked pred...


In [37]:
# Polarity + label: Hugging Face (RoBERTa). Subjectivity: TextBlob on raw comment text.
import time

start_time = time.time()
_p_neg, _p_neu, _p_pos, _pols, _labs = hf_sentiment_batch(texts)
df["hf_prob_negative"] = _p_neg
df["hf_prob_neutral"] = _p_neu
df["hf_prob_positive"] = _p_pos
df["polarity"] = _pols
df["label"] = _labs
df["subjectivity"] = [textblob_subjectivity(t) for t in texts]
print(
    f"HF sentiment ({HF_SENTIMENT_MODEL}) + TextBlob subjectivity: "
    f"{time.time() - start_time:.4f} seconds"
)
df.head()

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


HF sentiment (cardiffnlp/twitter-roberta-base-sentiment-latest) + TextBlob subjectivity: 279.8274 seconds


,comment,polarity,subjectivity,label,cleaned_comment,hf_prob_negative,hf_prob_neutral,hf_prob_positive
0,Climate change is the new military industrial ...,-0.726316,0.238636,negative,climate change new military industrial complex...,0.741378,0.243559,0.015063
1,"All good point, but wild fires aren’t because ...",-0.821760,0.566667,negative,good point wild fire climate change year fores...,0.836042,0.149676,0.014282
2,"Or maybe I do. \n \nFrom my comment, it's h...",-0.300549,0.583333,neutral,maybe comment hard tell since authority blame ...,0.361063,0.578424,0.060513
3,&gt; The people who make money from that disbe...,-0.269770,0.488492,neutral,gt people make money disbelief gt gt spend lot...,0.306249,0.657273,0.036479
4,All climate doomsday predictions up until now ...,-0.832769,0.633333,negative,climate doomsday prediction also debunked pred...,0.842093,0.148583,0.009324


In [38]:
# New workbook in 3.3/: original columns + HF softmax probs + derived polarity + TextBlob subjectivity + label
OUTPUT_XLSX = NB_DIR / "eval1kFinal_hf_sentiment.xlsx"
EVAL_SHEET = "eval1kFinal"

_out = df[_original_cols].copy()
_out["hf_prob_negative"] = df["hf_prob_negative"]
_out["hf_prob_neutral"] = df["hf_prob_neutral"]
_out["hf_prob_positive"] = df["hf_prob_positive"]
_out["polarity"] = df["polarity"]
_out["subjectivity"] = df["subjectivity"]
_out["label"] = df["label"]
_out.to_excel(OUTPUT_XLSX, sheet_name=EVAL_SHEET, index=False)
print(f"Wrote {OUTPUT_XLSX} (source unchanged: {file_path})")


Wrote /Users/weipingtee/Library/CloudStorage/OneDrive-NanyangTechnologicalUniversity/Year 3/Sem 2/SC4021 Information Retrieval/Assignment/SC4021/3.3/eval1kFinal_hf_sentiment.xlsx (source unchanged: /Users/weipingtee/Library/CloudStorage/OneDrive-NanyangTechnologicalUniversity/Year 3/Sem 2/SC4021 Information Retrieval/Assignment/SC4021/3.1new/3.1Final/evalPersonalCrawl.xlsx)
